# Earthquake Estimator · Análisis exploratorio de datos

Este cuaderno explora los registros sísmicos del catálogo USGS publicado en Kaggle. Antes de entrenar un modelo, queremos entender qué eventos contiene el archivo, qué información está disponible y qué limitaciones debemos tener en cuenta.

**Pregunta del proyecto:** ¿Podemos estimar la magnitud reportada de un sismo ya detectado a partir de su latitud, longitud y profundidad, y mejorar frente a usar la mediana histórica?

## 1. Carga y primera inspección del dataset

Cargamos `data/database.csv` con pandas. El archivo contiene **23.412 registros y 21 columnas**.

Usamos `df.head()` para ver las primeras cinco filas y reconocer cómo se registran la fecha, la hora, la ubicación, la profundidad y la magnitud. En esa muestra aparecen valores `NaN`, que representan datos faltantes. Esto nos lleva a revisar cuántos faltantes hay en cada columna del archivo completo.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

df = pd.read_csv(project_root / "data" / "database.csv")
df.head()

,Date,Time,Latitude,Longitude,Type,Depth,Depth Error,Depth Seismic Stations,Magnitude,Magnitude Type,...,Magnitude Seismic Stations,Azimuthal Gap,Horizontal Distance,Horizontal Error,Root Mean Square,ID,Source,Location Source,Magnitude Source,Status
0,01/02/1965,13:44:18,19.246,145.616,Earthquake,131.6,NaN,NaN,6.0,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860706,ISCGEM,ISCGEM,ISCGEM,Automatic
1,01/04/1965,11:29:49,1.863,127.352,Earthquake,80.0,NaN,NaN,5.8,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860737,ISCGEM,ISCGEM,ISCGEM,Automatic
2,01/05/1965,18:05:58,-20.579,-173.972,Earthquake,20.0,NaN,NaN,6.2,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860762,ISCGEM,ISCGEM,ISCGEM,Automatic
3,01/08/1965,18:49:43,-59.076,-23.557,Earthquake,15.0,NaN,NaN,5.8,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860856,ISCGEM,ISCGEM,ISCGEM,Automatic
4,01/09/1965,13:32:50,11.938,126.427,Earthquake,15.0,NaN,NaN,5.8,MW,...,NaN,NaN,NaN,NaN,NaN,ISCGEM860890,ISCGEM,ISCGEM,ISCGEM,Automatic


## 2. Conteo de valores faltantes por columna

Contamos cuántos valores faltan en cada columna del catálogo completo y ordenamos el resultado de mayor a menor. Así podemos ver dónde se concentra la información incompleta y comprobar si afecta a las variables que queremos usar para estimar la magnitud.

Este conteo es un diagnóstico inicial. Nos ayuda a decidir qué columnas revisar antes de preparar los datos para el modelo.

In [2]:
faltantes = df.isna().sum().sort_values(ascending=False)
print(faltantes.to_string())

Magnitude Error               23085
Horizontal Error              22256
Horizontal Distance           21808
Magnitude Seismic Stations    20848
Depth Error                   18951
Depth Seismic Stations        16315
Azimuthal Gap                 16113
Root Mean Square               6060
Magnitude Type                    3
Date                              0
Longitude                         0
Latitude                          0
Time                              0
Type                              0
Magnitude                         0
Depth                             0
ID                                0
Source                            0
Location Source                   0
Magnitude Source                  0
Status                            0


### Hallazgos del conteo de valores faltantes

- `Magnitude`, la variable que queremos estimar, tiene **0 valores faltantes**.
- `Latitude`, `Longitude` y `Depth`, las entradas propuestas para el modelo inicial, también tienen **0 valores faltantes**. Más adelante revisaremos si sus valores están dentro de rangos razonables.
- `Magnitude Error` y `Horizontal Error` concentran la mayor cantidad de faltantes, con **23.085** y **22.256**, respectivamente.

El siguiente paso es convertir estos conteos en porcentajes para saber qué proporción de los 23.412 registros está incompleta en cada columna.

## 3. Porcentaje de valores faltantes por columna

Dividimos el número de valores faltantes de cada columna entre los 23.412 registros del catálogo y multiplicamos por 100. Así podemos ver qué proporción de los registros está incompleta en cada columna.

Por ejemplo, los 23.085 faltantes de `Magnitude Error` representan el **98,60 %** del catálogo. Este resultado nos ayuda a decidir qué columnas requieren una revisión antes de usarlas.

In [3]:
porcentaje_faltantes = faltantes / len(df) * 100
print(porcentaje_faltantes.round(2).to_string())

Magnitude Error               98.60
Horizontal Error              95.06
Horizontal Distance           93.15
Magnitude Seismic Stations    89.05
Depth Error                   80.95
Depth Seismic Stations        69.69
Azimuthal Gap                 68.82
Root Mean Square              25.88
Magnitude Type                 0.01
Date                           0.00
Longitude                      0.00
Latitude                       0.00
Time                           0.00
Type                           0.00
Magnitude                      0.00
Depth                          0.00
ID                             0.00
Source                         0.00
Location Source                0.00
Magnitude Source               0.00
Status                         0.00


### Interpretación de los porcentajes de valores faltantes

- Cinco columnas superan el **80 % de valores faltantes**. Entre ellas, `Magnitude Error`, `Horizontal Error` y `Horizontal Distance` superan el **90 %**. Si las usáramos como entradas, tendríamos que rellenar la mayoría de sus valores o trabajar con muchos menos registros.
- Las entradas propuestas para el modelo inicial (`Latitude`, `Longitude` y `Depth`) y la variable objetivo (`Magnitude`) tienen **0 % de valores faltantes**. Podemos conservarlas para el análisis sin imputar datos en esas columnas. Aún debemos revisar si sus valores están dentro de rangos razonables.
- `Magnitude Type` tiene solo **3 valores faltantes**, equivalentes al **0,01 %**. Podemos usar esta columna para describir los tipos de magnitud reportados en el catálogo.

Por ahora, el CSV se conserva completo. Estos resultados documentan la disponibilidad de los datos y orientan la selección inicial de variables.

## 4. Tipos de eventos del catálogo

Revisamos la columna `Type` para identificar qué tipos de eventos contiene el catálogo y cuántos registros hay de cada uno.

Como el proyecto se enfoca en terremotos, este conteo nos permitirá saber cuántas filas corresponden a `Earthquake` y cuántas quedarán fuera del análisis cuando apliquemos el filtro. Primero contamos los eventos de cada tipo para documentar el criterio de selección.

In [4]:
conteo_eventos = df["Type"].value_counts()
print(conteo_eventos.to_string())

Type
Earthquake           23232
Nuclear Explosion      175
Explosion                4
Rock Burst               1


### Hallazgos por tipo de evento

De los **23.412 registros** del catálogo, **23.232** están clasificados como `Earthquake` y representan el **99,23 %** del total. Los **180 registros restantes** corresponden a 175 explosiones nucleares, 4 explosiones y 1 evento clasificado como `Rock Burst`.

Como el objetivo del proyecto es estimar la magnitud de terremotos, en el siguiente paso seleccionaremos las filas con `Type == "Earthquake"`. Trabajaremos con esa selección en una tabla separada para conservar el catálogo completo como referencia.

In [5]:
terremotos = df.loc[df["Type"] == "Earthquake"].copy()

print(f"Registros del catálogo: {len(df)}")
print(f"Terremotos seleccionados: {len(terremotos)}")
print(f"Otros eventos excluidos: {len(df) - len(terremotos)}")


Registros del catálogo: 23412
Terremotos seleccionados: 23232
Otros eventos excluidos: 180


### Resultado del filtro y alcance del análisis

El catálogo original contiene **23.412 registros**. Al seleccionar los eventos clasificados como `Earthquake`, obtuvimos **23.232 terremotos** y excluimos **180 eventos** de otros tipos.

Conservamos el catálogo original en `df` y guardamos la selección en `terremotos`. Esta segunda tabla será la base para explorar la distribución de magnitudes, la ubicación de los epicentros y la relación entre profundidad y magnitud.

El diagnóstico de valores faltantes presentado arriba corresponde al catálogo completo. A partir de este punto, los resultados del análisis se referirán a los terremotos seleccionados.